# WGDSA in a Multigroup HDPE Shielding Problem

![Multigroup source geometry in an HDPE moderator](images/wgdsa_hdpe_geometry.png)

This tutorial compares source iteration with and without within-group diffusion synthetic acceleration (WGDSA) in a heterogeneous two-dimensional shielding problem. A compact volumetric source is embedded in a strongly scattering HDPE moderator. This standalone variant removes the He-3 detector from the original source--detector problem and fills its former region with HDPE.

## Reduced multigroup model

The geometry has a 2 cm square source centered near the lower-left corner; every other cell is HDPE. The HDPE and Cf-252 data were reduced from the 69-group WIMS tutorial library to 12 groups. Total and absorption cross sections use an unweighted fine-group average, while transfer terms are averaged over source groups and summed over destination groups. This makes the data suitable for demonstrating solver behavior, but not for design analysis. The source spectrum is a similarly collapsed Cf-252 Watt spectrum.

The mesh uses a 2-by-2 KBA decomposition, so run the generated script with four MPI processes. Sixteen directions keep the comparison inexpensive while retaining a genuine parallel transport sweep.

In [ ]:
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
num_groups = 12
source_spectrum = [
    8.70673882298e-1, 1.27603333571e-1, 1.69874391340e-3,
    2.39751217396e-5, 6.37406413603e-8, 8.61581497551e-10,
    1.06916189569e-10, 2.94232444417e-10, 5.81630205686e-11,
    2.77856961803e-11, 4.45769378270e-12, 1.77083905175e-12,
]

## Mesh and materials

Block 0 is the HDPE moderator and block 2 is the source region. The source logical volume makes the region independent of external mesh files or command-line parameters.

In [ ]:
def load_xs(filename):
    xs = MultiGroupXS()
    xs.LoadFromOpenSn(str(tutorial_dir / filename))
    return xs


def make_mesh():
    nodes = [0.5 * i for i in range(41)]
    partitioner = KBAGraphPartitioner(
        nx=2, ny=2, xcuts=[10.0], ycuts=[10.0]
    )
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes], partitioner=partitioner
    ).Execute()
    mesh.SetOrthogonalBoundaries()
    mesh.SetUniformBlockID(0)

    source = RPPLogicalVolume(
        xmin=2.0, xmax=4.0, ymin=2.0, ymax=4.0, infz=True
    )
    mesh.SetBlockIDFromLogicalVolume(source, 2, True)
    return mesh

## Build one solver with optional WGDSA

Each energy group has its own groupset. This preserves the fast-to-thermal downscatter ordering while allowing WGDSA to target the slowly converging spatial error within every group. Both cases use classic Richardson source iteration so the effect of the diffusion correction is visible.

The reported sweep count is the total over all groupsets. Wall time is split into three phases, each the maximum across ranks: model setup (cross sections, mesh, and transport problem, including the WGDSA diffusion operators that OpenSn builds with the problem), solver initialization, and the source iterations themselves.

In [ ]:
def solve(use_wgdsa):
    comm.Barrier()
    start = MPI.Wtime()
    moderator_xs = load_xs("wgdsa_multigroup_moderator.cxs")
    source_xs = load_xs("wgdsa_multigroup_source.cxs")

    groupset_options = {
        "angular_quadrature": GLCProductQuadrature2DXY(
            n_polar=2, n_azimuthal=16, scattering_order=0
        ),
        "inner_linear_method": "classic_richardson",
        "l_abs_tol": 1.0e-8,
        "l_max_its": 1000,
    }
    if use_wgdsa:
        groupset_options.update(
            {
                "apply_wgdsa": True,
                "wgdsa_l_abs_tol": 1.0e-8,
                "wgdsa_l_max_its": 200,
                "wgdsa_solver_policy": "auto",
            }
        )

    groupsets = [
        dict(groupset_options, groups_from_to=(group, group))
        for group in range(num_groups)
    ]
    problem = DiscreteOrdinatesProblem(
        mesh=make_mesh(),
        num_groups=num_groups,
        groupsets=groupsets,
        xs_map=[
            {"block_ids": [0], "xs": moderator_xs},
            {"block_ids": [2], "xs": source_xs},
        ],
        volumetric_sources=[
            VolumetricSource(block_ids=[2], group_strength=source_spectrum)
        ],
        boundary_conditions=[
            {"name": "xmin", "type": "vacuum"},
            {"name": "xmax", "type": "vacuum"},
            {"name": "ymin", "type": "vacuum"},
            {"name": "ymax", "type": "vacuum"},
        ],
        options={
            "max_ags_iterations": 20,
            "ags_tolerance": 1.0e-8,
            "ags_convergence_check": "pointwise",
            "verbose_inner_iterations": False,
        },
    )

    solver = SteadyStateSourceSolver(problem=problem)

    comm.Barrier()
    model_done = MPI.Wtime()
    solver.Initialize()
    comm.Barrier()
    init_done = MPI.Wtime()
    solver.Execute()
    comm.Barrier()
    end = MPI.Wtime()
    times = [
        comm.allreduce(t, op=MPI.MAX)
        for t in (model_done - start, init_done - model_done, end - init_done)
    ]

    average = VolumePostprocessor(problem=problem, value_type="avg")
    average.Execute()
    total_average_flux = sum(average.GetValue()[0])
    return problem, total_average_flux, solver.GetNumSweeps(), times

## Compare the solution, sweep count, and runtime

WGDSA changes the convergence path, not the converged transport solution. Agreement in the energy-summed, volume-averaged scalar flux is therefore the correctness check. Transport sweeps measure convergence work, while wall time shows the practical effect of the acceleration. Timing is printed rather than asserted because it depends on the machine and MPI configuration. Reporting the setup and iteration phases separately isolates the cost and benefit of WGDSA from the setup both cases share.

In [ ]:
_, unaccelerated_flux, unaccelerated_sweeps, unaccelerated_times = solve(False)
wgdsa_problem, wgdsa_flux, wgdsa_sweeps, wgdsa_times = solve(True)
relative_difference = abs(wgdsa_flux - unaccelerated_flux) / abs(unaccelerated_flux)
iteration_speedup = unaccelerated_times[2] / wgdsa_times[2]
total_speedup = sum(unaccelerated_times) / sum(wgdsa_times)

if rank == 0:
    print(f"Unaccelerated total average flux={unaccelerated_flux:.12e}")
    print(f"WGDSA total average flux={wgdsa_flux:.12e}")
    print(f"WGDSA total flux relative difference={relative_difference:.12e}")
    print(f"Unaccelerated sweeps={unaccelerated_sweeps}")
    print(f"WGDSA sweeps={wgdsa_sweeps}")
    for name, times, sweeps in (
        ("Unaccelerated", unaccelerated_times, unaccelerated_sweeps),
        ("WGDSA", wgdsa_times, wgdsa_sweeps),
    ):
        print(f"{name} model setup time (s)={times[0]:.6f}")
        print(f"{name} solver initialization time (s)={times[1]:.6f}")
        print(f"{name} iteration time (s)={times[2]:.6f}")
        print(f"{name} total wall time (s)={sum(times):.6f}")
        print(f"{name} time per sweep (s)={times[2] / sweeps:.6f}")
    print(f"WGDSA iteration speedup={iteration_speedup:.6f}")
    print(f"WGDSA total speedup={total_speedup:.6f}")

assert relative_difference < 1.0e-6
assert wgdsa_sweeps < unaccelerated_sweeps

A representative four-process run gives the following comparison:

| Solve | Energy-summed average flux | Transport sweeps |
|---|---:|---:|
| No acceleration | 0.162415762 | 5394 |
| WGDSA | 0.162415766 | 1929 |

The scalar fluxes agree to approximately $2.5\times10^{-8}$ relative difference, and WGDSA reduces the transport-sweep count by about 64%. The sweep counts are deterministic for this configuration.

The wall-time breakdown below is the median of five four-process runs:

| Solve | Model setup (s) | Solver initialization (s) | Source iterations (s) | Total (s) | Time per sweep (ms) |
|---|---:|---:|---:|---:|---:|
| No acceleration | 0.026 | <0.001 | 5.933 | 5.958 | 1.10 |
| WGDSA | 0.067 | <0.001 | 3.000 | 3.067 | 1.56 |

Setup is small in both cases, about 1% and 2% of the total. WGDSA's model setup is about 0.04 s longer because the problem builds and assembles a diffusion operator for each of the 12 groupsets. During the iterations, each within-group diffusion correction adds about 0.46 ms to a sweep (about 41%), roughly 0.9 s over the whole solve. The 2.8-fold reduction in sweeps still dominates, giving a speedup of 1.97 for the iterations and 1.93 overall. Wall time is machine-dependent.

## Export and visualize the scalar flux

The two solutions agree, so the WGDSA result is used as the representative flux field. The following code exports the 12 group scalar fluxes to VTK. It remains in Markdown so regression tests do not create output files.

```python
from pyopensn.fieldfunc import FieldFunctionGridBased

FieldFunctionGridBased.ExportMultipleToPVTU(
    wgdsa_problem.GetScalarFluxFieldFunction(), "Flux/WGDSA_Flux"
)
```

![Energy-summed and thermal-group scalar flux from the HDPE-only WGDSA solve](images/wgdsa_hdpe_flux.png)

*Scalar flux from the WGDSA solve on a logarithmic scale: the sum over all 12 groups (left) and the thermal group 11 (right). Each 0.5 cm cell is drawn with its piecewise-linear discontinuous (PWLD) solution, the bilinear field defined by the cell's four nodal values. The dashed square marks the source. The former detector region is filled with the same HDPE as the rest of the moderator.*

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()